# 华润江中 600750 — 双均线策略 MA10xMA60 完整回测

**策略逻辑：**
- MA{FAST} 上穿 MA{SLOW} → 金叉买入
- MA{FAST} 下穿 MA{SLOW} → 死叉卖出
- 数据已做前复权处理

## 1. 环境准备

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

COLOR_UP, COLOR_DOWN = '#ef5350', '#26a69a'
print('OK')

OK


## 2. 加载复权后股价数据

In [2]:
df = pd.read_csv("600750_daily_fq.csv")
df["trade_date"] = pd.to_datetime(df["trade_date"], format="%Y%m%d")
df = df.sort_values("trade_date").reset_index(drop=True)
close = df["close"].values; dates = df["trade_date"]
print(f"加载: {len(df)} 交易日, {dates.iloc[0].strftime("%Y-%m-%d")} ~ {dates.iloc[-1].strftime("%Y-%m-%d")}")
df.head()

加载: 241 交易日, 2025-06-30 ~ 2026-06-26


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount
0,600750.SH,2025-06-30,20.65,20.72,20.52,20.57,20.64,-0.07,-0.3391,38656.20,84419.665
1,600750.SH,2025-07-01,20.57,20.63,20.49,20.57,20.57,0.00,0.0000,37937.60,82782.419
2,600750.SH,2025-07-02,20.60,20.69,20.51,20.59,20.57,0.02,0.0972,37542.50,81945.726
3,600750.SH,2025-07-03,20.63,20.87,20.59,20.71,20.59,0.12,0.5828,60048.97,132069.199
4,600750.SH,2025-07-04,20.71,20.74,20.63,20.72,20.71,0.01,0.0483,37063.13,81406.499


## 3. 计算 MA10 x MA60 与交易信号

In [3]:
FAST, SLOW = 10, 60
df["ma_fast"] = df["close"].rolling(window=FAST).mean()
df["ma_slow"] = df["close"].rolling(window=SLOW).mean()
df["signal"] = 0
golden = (df["ma_fast"].shift(1) <= df["ma_slow"].shift(1)) & (df["ma_fast"] > df["ma_slow"])
death  = (df["ma_fast"].shift(1) >= df["ma_slow"].shift(1)) & (df["ma_fast"] < df["ma_slow"])
df.loc[golden, "signal"] = 1; df.loc[death, "signal"] = -1
in_position = False
for i in range(len(df)):
    if df.loc[i, "signal"] == 1: in_position = True
    elif df.loc[i, "signal"] == -1: in_position = False
    df.loc[i, "position"] = int(in_position)
print(f"金叉 {(df["signal"]==1).sum()} 次, 死叉 {(df["signal"]==-1).sum()} 次")
df[["trade_date","close","ma_fast","ma_slow","signal","position"]].tail(12)

金叉 4 次, 死叉 4 次


,trade_date,close,ma_fast,ma_slow,signal,position
229,2026-06-10,24.64,24.609,24.387667,0,1.0
230,2026-06-11,24.10,24.734,24.383000,0,1.0
231,2026-06-12,23.67,24.687,24.372833,0,1.0
232,2026-06-15,23.81,24.574,24.360500,0,1.0
233,2026-06-16,23.35,24.401,24.336167,0,1.0
234,2026-06-17,23.38,24.223,24.315000,-1,0.0
235,2026-06-18,23.43,24.107,24.269333,0,0.0
236,2026-06-22,23.35,23.944,24.244667,0,0.0
237,2026-06-23,23.79,23.821,24.229833,0,0.0
238,2026-06-24,23.09,23.661,24.197167,0,0.0


## 4. 模拟交易与回测

In [4]:
initial_capital = 100000.0; capital = initial_capital; shares = 0
portfolio_values = []; daily_returns = []
for i in range(len(df)):
    price = df.loc[i, "close"]
    if i >= SLOW:
        if df.loc[i, "signal"] == 1 and capital > 0:
            shares = capital / price; capital = 0
        elif df.loc[i, "signal"] == -1 and shares > 0:
            capital = shares * price; shares = 0
    tv = capital + shares * price
    portfolio_values.append(tv)
    daily_returns.append(0 if i == 0 else (tv - portfolio_values[-2]) / portfolio_values[-2])
df["portfolio_value"] = portfolio_values; df["daily_return"] = daily_returns
print(f"初始: {initial_capital:,.0f} → 最终: {portfolio_values[-1]:,.2f}")

初始: 100,000 → 最终: 98,821.82


## 5. 计算七大核心量化指标

In [5]:
total_days = (dates.iloc[-1] - dates.iloc[0]).days
years = total_days / 365.25; annual_rf = 0.015
total_return = (portfolio_values[-1] - initial_capital) / initial_capital
cagr = (portfolio_values[-1] / initial_capital) ** (1 / years) - 1
buy_hold_cagr = (close[-1] / close[0]) ** (1 / years) - 1
alpha = cagr - buy_hold_cagr
peak_val = portfolio_values[0]; max_dd = 0
for v in portfolio_values:
    if v > peak_val: peak_val = v
    dd = (peak_val - v) / peak_val
    if dd > max_dd: max_dd = dd
ret_s = pd.Series(daily_returns).iloc[SLOW:]
win_rate = (ret_s > 0).sum() / len(ret_s)
pos = ret_s[ret_s > 0]; neg = ret_s[ret_s < 0]
pl_ratio = pos.mean() / abs(neg.mean()) if len(neg) > 0 else 999
sharp_ret = ret_s.mean() * 252; sharp_std = ret_s.std() * np.sqrt(252)
sharpe = (sharp_ret - annual_rf) / sharp_std if sharp_std > 0 else 0
print(f'总收益率: {total_return*100:+6.2f}%')
print(f'年化收益率: {cagr*100:+6.2f}%')
print(f'超额收益: {alpha*100:+6.2f}%')
print(f'最大回撤: {max_dd*100:6.2f}%')
print(f'胜率: {win_rate*100:6.1f}%')
print(f'盈亏比: {pl_ratio:6.2f}')
print(f'夏普比率: {sharpe:6.2f}')
print(f'波动率: {sharp_std*100:6.1f}%')

总收益率:  -1.18%
年化收益率:  -1.19%
超额收益: -10.30%
最大回撤:  17.40%
胜率:   34.8%
盈亏比:   1.09
夏普比率:  -0.02
波动率:   22.7%


## 6. 可视化：股价·双均线·信号·资金曲线

In [6]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    row_heights=[0.65, 0.35],
    subplot_titles=("股价 · 双均线 · 交易信号", "资金曲线 & 回撤"))
buy_dates = df[df['signal'] == 1]; sell_dates = df[df['signal'] == -1]

fig.add_trace(go.Candlestick(x=dates, open=df['open'], high=df['high'], low=df['low'],
    close=df['close'], name='K线', increasing_line_color=COLOR_UP,
    decreasing_line_color=COLOR_DOWN), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['ma_fast'], mode='lines',
    line=dict(color='#2196f3', width=1.5), name=f'MA10'), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['ma_slow'], mode='lines',
    line=dict(color='#ff9800', width=1.5), name=f'MA60'), row=1, col=1)
fig.add_trace(go.Scatter(x=buy_dates['trade_date'], y=buy_dates['low']*0.985,
    mode='markers', marker=dict(symbol='triangle-up', size=12, color=COLOR_UP),
    name='买入 (金叉)'), row=1, col=1)
fig.add_trace(go.Scatter(x=sell_dates['trade_date'], y=sell_dates['high']*1.015,
    mode='markers', marker=dict(symbol='triangle-down', size=12, color=COLOR_DOWN),
    name='卖出 (死叉)'), row=1, col=1)
fig.add_trace(go.Scatter(x=dates, y=df['portfolio_value'], mode='lines',
    line=dict(color='#7c4dff', width=2), name='策略净值',
    fill='tozeroy', fillcolor='rgba(124,77,255,0.06)'), row=2, col=1)
fig.add_hline(y=initial_capital, line_dash='dot', line_color='rgba(0,0,0,0.25)',
    row=2, col=1, annotation_text='本金')

fig.update_layout(
    title=f"<b>600750 MA10xMA60 双均线策略回测</b><br>"
          f"<sup>总收益 -1.2% | CAGR -1.2% | 夏普 -0.02</sup>",
    xaxis_rangeslider_visible=False, height=900, hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    template='plotly_white')
fig.update_xaxes(rangeselector=dict(buttons=list([
    dict(count=1, label='1月', step='month', stepmode='backward'),
    dict(count=3, label='3月', step='month', stepmode='backward'),
    dict(step='all', label='全部')])))
fig.update_yaxes(title_text='价格', row=1, col=1)
fig.update_yaxes(title_text='资金曲线', row=2, col=1)
fig.show()

## 7. 回测结论

### 七大指标汇总

| 类别 | 指标 | 数值 |
|------|------|------|
| 收益 | 总收益率 | -1.18% |
| 收益 | 年化收益率 CAGR | -1.19% |
| 收益 | 超额收益 Alpha | -10.30% (vs 买入持有 +9.1%) |
| 风险 | 最大回撤 MDD | 17.40% |
| 交易质量 | 胜率 | 34.8% |
| 交易质量 | 盈亏比 | 1.09 |
| 综合 | 夏普比率 | -0.02 |

### 回测概要
- 策略: MA10 × MA60 双均线金叉买入/死叉卖出
- 区间: 2025-06-30 ~ 2026-06-26 (1.0 年)
- 信号: 金叉 4 次, 死叉 4 次
- 持仓占比: 53.9%
- 基准买入持有: +9.0%

> **注：** 本回测未考虑交易手续费、滑点、印花税等成本。数据已做前复权处理。
